[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lmassaron/finetuning/blob/main/chapter_08/listing_8.1.ipynb)

In [1]:
import sys
if "google.colab" in sys.modules:
    !pip install -q -U sentence-transformers "torchao>=0.16.0" bitsandbytes peft datasets transformers


### Listing 6.17: Library Imports

In [2]:
import torch
import warnings
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig,
    GenerationConfig
)
import random
import numpy as np
from datasets import load_dataset
from transformers import AutoConfig
from sentence_transformers import SentenceTransformer
from peft import get_peft_model, LoraConfig, TaskType
from sentence_transformers.sentence_transformer.losses import (
    MultipleNegativesRankingLoss,)
from sentence_transformers.sentence_transformer.training_args import (
    SentenceTransformerTrainingArguments,)
from sentence_transformers.sentence_transformer.trainer import (
    SentenceTransformerTrainer,)

warnings.filterwarnings("ignore")

### Listing 6.18: Loading and Sampling the Synthetic Retrieval Dataset

In [3]:
print("Loading dataset...")
raw_ds = load_dataset("nvidia/Retrieval-Synthetic-NVDocs-v1",
                      split="train[:5000]")

sample_ds = (raw_ds.shuffle(seed=42)
                        .select(range(min(3000, len(raw_ds))))
)

print(f"Sampled {len(sample_ds)} random examples.")

Loading dataset...
Sampled 3000 random examples.


### Listing 6.19: Configuring and Loading the LLM Text Generation Pipeline

In [4]:
qwen_bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
llm_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading {llm_id}...")
tokenizer = AutoTokenizer.from_pretrained(llm_id, clean_up_tokenization_spaces=False)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm_model = AutoModelForCausalLM.from_pretrained(
    llm_id, quantization_config=qwen_bnb, device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer,
)

pipe.generation_config = GenerationConfig(
    max_new_tokens=50,
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id,
)

Loading Qwen/Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

### Listing 6.20: Defining the Document Chunking and Question Generation Function

In [5]:
def generate_synthetic_pair(example, chunk_length=1500):

    doc = str(example.get("text", "")).strip()
    if len(doc) > chunk_length:
        random_start = random.randint(0, len(doc) - chunk_length)
        start_pos = doc.find(". ", random_start)
        start_idx = start_pos + 2 if start_pos != -1 else random_start
        end_pos = doc.find(".", start_idx + chunk_length)
        end_idx = end_pos + 1 if end_pos != -1 else len(doc)
        chunk = doc[start_idx:end_idx].strip()
    else:
        chunk = doc

    prompt = f"<|im_start|>user\nRead the following document chunk and generate a single, short, specific question that is directly answered by it.\n\nDocument: {chunk}\n\nQuestion:<|im_end|>\n<|im_start|>assistant\n"

    outputs = pipe(
        prompt,
        truncation=True,
        return_full_text=False,
    )
    question = outputs[0]["generated_text"].strip()

    return {"anchor": question, "positive": chunk}

### Listing 6.21: Splitting the Dataset and Batch Generating Synthetic Pairs

In [6]:
split_sample_ds = sample_ds.train_test_split(test_size=0.2, seed=42)
train_docs = split_sample_ds["train"]
test_docs = split_sample_ds["test"]

print(f"Splitting documents: {len(train_docs)} train docs | {len(test_docs)} eval docs")
print("Generating synthetic pairs for training documents...")
train_ds = train_docs.map(generate_synthetic_pair, remove_columns=train_docs.column_names)
train_ds = train_ds.shuffle(seed=42)

print("Generating synthetic pairs for evaluation documents...")
eval_ds = test_docs.map(generate_synthetic_pair, remove_columns=test_docs.column_names)
eval_ds = eval_ds.shuffle(seed=42)

print(f"\nGeneration Complete! Generated {len(train_ds)} train pairs and {len(eval_ds)} eval pairs.")
print("Here are 3 training examples:")
for i in range(min(3, len(train_ds))):
    print(f"\n--- Example {i + 1} ---")
    print("Q:", train_ds[i]["anchor"])
    print("A:", train_ds[i]["positive"][:200], "...")


Splitting documents: 2400 train docs | 600 eval docs
Generating synthetic pairs for training documents...


Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Generating synthetic pairs for evaluation documents...


Map:   0%|          | 0/600 [00:00<?, ? examples/s]


Generation Complete! Generated 2400 train pairs and 600 eval pairs.
Here are 3 training examples:

--- Example 1 ---
Q: What does dSPACE's modeling capabilities enable?
A: However, to be effective, it must be able to translate to real-world driving.
dSPACE s modeling capabilities are key to understanding vehicle behavior in diverse conditions, enabling the exhaustive an ...

--- Example 2 ---
Q: What are some examples of AI agents being deployed in smart cities?
A: Then, K2K uses the data to fine-tune the VLMs powering the AI agents with NeMo Curator. These simulations enable K2K s AI agents to create over 100,000 predictions per second.
Milestone Systems in col ...

--- Example 3 ---
Q: What industries does NVIDIA serve in?
A: Our partners are here to assist your organization at every level to build and execute transformative AI strategies, products, and services.
Meet Our Partners

Get Started



Take the Next Steps


Stay ...


### Listing 6.22: Loading the Base Embedding Model and Applying LoRA Adapters

In [7]:
embed_id = "nvidia/Nemotron-3-Embed-1B-BF16"

print(f"Loading base embedding model {embed_id}...")
embed_model = SentenceTransformer(
    embed_id, model_kwargs={"dtype": torch.bfloat16}, 
    )

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION,)

peft_model = get_peft_model(embed_model[0].auto_model, lora_config)
embed_model[0].auto_model = peft_model
print("\nLoRA trainable parameters:")
peft_model.print_trainable_parameters()

Loading base embedding model nvidia/Nemotron-3-Embed-1B-BF16...


Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}
Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}



LoRA trainable parameters:
trainable params: 5,242,880 || all params: 1,146,161,152 || trainable%: 0.4574


### Listing 6.23: Configuring Multiple Negatives Ranking Loss and Executing Training

In [8]:
loss = MultipleNegativesRankingLoss(embed_model)

args = SentenceTransformerTrainingArguments(
    output_dir="./nemotron-embed-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=50,
    save_strategy="no",
    report_to="none",)

trainer = SentenceTransformerTrainer(
    model=embed_model, args=args, train_dataset=train_ds, loss=loss)

trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
50,0.119226
100,0.057165
150,0.020466
200,0.009411
250,0.066602
300,0.007126
350,0.034555
400,0.010687
450,0.008861
500,0.013097


TrainOutput(global_step=1200, training_loss=0.02572288528084755, metrics={'train_runtime': 253.1402, 'train_samples_per_second': 9.481, 'train_steps_per_second': 4.74, 'total_flos': 0.0, 'train_loss': 0.02572288528084755, 'epoch': 1.0})

### Listing 6.24: Evaluating Retrieval Performance Using Strict Recall@1

In [9]:
eval_queries = eval_ds["anchor"]
eval_docs = eval_ds["positive"]

def evaluate_retrieval(model, model_name):
    query_embs = model.encode(eval_queries)
    doc_embs = model.encode(eval_docs)
    similarities = np.dot(query_embs, doc_embs.T)
    top_doc_indices = np.argmax(similarities, axis=1)
    correct_indices = np.arange(len(eval_queries))
    correct_retrievals = np.sum(top_doc_indices == correct_indices)
    recall_at_1 = correct_retrievals / len(eval_queries)
    print(f"{model_name} Strict Recall@1: {recall_at_1 * 100:.2f}%")
    return recall_at_1

print("\n--- Evaluation Results ---")

original_model = SentenceTransformer(
    embed_id, model_kwargs={"dtype": torch.bfloat16})
finetuned_model = embed_model

original_score = evaluate_retrieval(original_model, "Original Model")
finetuned_score = evaluate_retrieval(finetuned_model, "Fine-Tuned Model")

improvement = finetuned_score - original_score
print(f"Absolute Improvement: +{improvement * 100:.2f}%")


--- Evaluation Results ---


Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}
Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}


Original Model Strict Recall@1: 33.83%
Fine-Tuned Model Strict Recall@1: 68.67%
Absolute Improvement: +34.83%
